## Plots for the figures of the paper (revolving only simulations)

In [ ]:
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches

plt.rcParams["pdf.fonttype"] = 42

from IPython.display import HTML

import file_funcs
import plot_funcs

import colors

colors_lst, red, custom_cmap = colors.color_scheme(scheme="Leon", add_shim=False)

## Success matrices - pos force and combined diagrams

In [ ]:
import copy
import numpy as np
from matplotlib import patches

import colors

importlib.reload(file_funcs)

H = 4
N = 2**H

folder_pos = Path("Training\\May17Like_noFlipChain")
folder_force = Path("Training\\Apr23randomPosAfterFroceExplode")

M_pos, _, _ = file_funcs.build_success_matrix(
    folder_pos, N=N, old=False, near_miss=False, symmetry=False,
    omit_inverted=True, find_symmetrical=False,
)
M_F, _, _ = file_funcs.build_success_matrix(
    folder_force, N=N, old=False, near_miss=False, symmetry=False,
    omit_inverted=True, find_symmetrical=False,
)

M_pos_includeSymm = copy.copy(M_pos)
M_pos_includeSymm[M_pos[::-1, ::-1] == 0] = 0

M_F_includeSymm = copy.copy(M_F)
M_F_includeSymm[M_F[::-1, ::-1] == 0] = 0

# Both position and force
M_both = copy.copy(M_pos_includeSymm)
M_both[M_F_includeSymm == 0] = 0
M_both[M_both[::-1, ::-1] == 0] = 0


In [ ]:

labels = [format(i, f"0{H}b") for i in range(N)]
_, _, custom_cmap = colors.color_scheme()
font_size = 18

fig, axes = plt.subplots(
    1, 3, figsize=(13.5, 4.8), sharey=True, constrained_layout=True,
)
for ax, matrix, title in zip(
    axes,
    (M_pos_includeSymm, M_F_includeSymm, M_both),
    ("Position", "Force", "Combined"),
):
    matrix_masked = np.ma.masked_where(np.eye(N, dtype=bool), matrix)
    image = ax.imshow(
        matrix_masked, cmap=custom_cmap, vmin=0, vmax=4,
        origin="lower", interpolation="none", aspect="equal",
    )
    ax.set_xticks(range(N), labels, rotation=90)
    ax.set_yticks(range(N), labels)
    ax.set_xlabel("desired buckle", fontsize=font_size)
    ax.set_title(title, fontsize=font_size, pad=9)
    ax.tick_params(axis="both", labelsize=font_size)

axes[0].set_ylabel("initial buckle", fontsize=font_size)
for ax in axes[1:]:
    ax.tick_params(axis="y", left=False, labelleft=False)

legend_handles = [
    patches.Patch(facecolor=custom_cmap(image.norm(0)), label="Success"),
    # patches.Patch(facecolor=custom_cmap(image.norm(1)), label="Missing"),
    patches.Patch(facecolor=custom_cmap(image.norm(2)), label="Failure"),
]
axes[-1].legend(
    handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
    fontsize=font_size, frameon=False, borderaxespad=0,
)
fig.savefig("success_mat.pdf", format="pdf", bbox_inches="tight")
plt.show()

success_rate_both = np.sum(M_both == 0) / (np.sum(M_both == 2) + np.sum(M_both == 0))
print("success_rate_both=", success_rate_both)


## Efficiently covering transitions

In [ ]:
# Hamming-transition coverage from exported training or tip-sweep files
from pathlib import Path
import pandas as pd

importlib.reload(file_funcs)

folder_pos = Path("efficient_transitions\\May17Like_noFlipChain")
folder_F = Path("efficient_transitions\\Apr23randomPosAfterFroceExplode")
folder_rand = Path("efficient_transitions\\Aug1_grid_sweep_arc")

coverage_pos = pd.read_csv(folder_pos / "cumulative_transition.csv")
coverage_F = pd.read_csv(folder_F / "cumulative_transition.csv")
coverage_rand = pd.read_csv(folder_rand / "cumulative_transition.csv")


In [ ]:
importlib.reload(plot_funcs)

export_png = True
export_pdf = True
save_stem = Path("Figures") / "cumulative_transition"

fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(8, 4), sharey=True,
    gridspec_kw={"width_ratios": (3, 1), "wspace": 0.06},
)
for coverage_df, x_col, label, color_index in (
    (coverage_pos, "training_task", "Position", 0),
    (coverage_F, "training_task", "Force", 1),
    (coverage_rand, "step", "Random", 2),
):
    for ax in (ax_left, ax_right):
        plot_funcs.plot_cumulative_transition_curve(
            coverage_df,
            x_col=x_col,
            x_label="",
            label=label if ax is ax_left else None,
            color_index=color_index,
            ax=ax,
        )

ax_left.set_xlim(0, 240)
ax_right.set_xlim(2260, 2310)  # The random sweep currently ends at step 2304.
ax_left.set_xticks([0, 40, 80, 120, 160, 200])
ax_right.set_xticks([2260, 2290, 2310])
ax_left.spines["right"].set_visible(False)
ax_right.spines["left"].set_visible(False)
ax_right.set_ylabel("")
ax_right.tick_params(axis="y", left=False, labelleft=False)

# Diagonal marks indicate the omitted x range.
break_size = 0.018
break_style = dict(color="k", clip_on=False, linewidth=1.2)
ax_left.plot((1 - break_size, 1 + break_size), (-break_size, +break_size),
             transform=ax_left.transAxes, **break_style)
ax_right.plot((-break_size, +break_size), (-break_size, +break_size),
              transform=ax_right.transAxes, **break_style)
fig.supxlabel("training task / sweep step", y=0.01)

fig.subplots_adjust(bottom=0.18)
if export_png:
    fig.savefig(save_stem.with_suffix(".png"), dpi=200, bbox_inches="tight")
if export_pdf:
    with plt.rc_context({"pdf.fonttype": 42}):
        fig.savefig(save_stem.with_suffix(".pdf"), format="pdf", bbox_inches="tight")
plt.show()

## Transitions - init to fin

In [ ]:
importlib.reload(file_funcs)
import helpers_builders
importlib.reload(helpers_builders)

folder_force = Path("Training\\Apr23randomPosAfterFroceExplode")
# folder = Path("Training\\May17Pos_symmetrical_delta")
folder_pos = Path("Training\\May17Like_noFlipChain")
folder_sweep = Path("grid sweep\\grid\\tip_grid_transition_csvs_alternating_angle_all_steps")
# folder_sweep = Path("grid sweep\\restart_every_t")
only_reached_nodes = False
only_init_and_final_buckles = True
omit_inverted = True
transition_mode = "ring"  # "hamming" for one-bit missing edges; "ring" for all-to-all missing edges
reciprocity = True  # Add each transition count to the bitwise sign-opposite transition
transitions_pos, _, _, edge_zero_loss_count_pos, missing_edges_pos = file_funcs.buckle_transitions(folder_pos,
                                                                                                   only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                                   omit_inverted = omit_inverted,
                                                                                                   transition_mode=transition_mode,
                                                                                                   reciprocity=reciprocity)
transitions_F, _, _, edge_zero_loss_count_F, missing_edges_F = file_funcs.buckle_transitions(folder_force,
                                                                                             only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                             omit_inverted = omit_inverted,
                                                                                             transition_mode=transition_mode,
                                                                                             reciprocity=reciprocity)
transitions_swp, _, _, edge_zero_loss_count_swp, missing_edges_swp = file_funcs.buckle_transitions(folder_sweep,
                                                                                                   only_init_and_final_buckles=only_init_and_final_buckles,
                                                                                                   omit_inverted = omit_inverted,
                                                                                                   transition_mode=transition_mode,
                                                                                                   reciprocity=reciprocity)


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

H = 4
N = 2**H
font_size=20
save = "pdf"
plot = "both"

scheme = "Leon"
if scheme == "Leon":
    color_intentional = colors_lst[3]
    color_unintentional = colors_lst[2]
elif scheme == "mine":
    color_intentional = colors_lst[1]
    color_unintentional = colors_lst[0]

# colors_lst, _, _ = colors.color_scheme()
transition_cmap = ListedColormap([red, color_unintentional, color_intentional])
transition_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], transition_cmap.N)
labels = [format(i, f"0{H}b") for i in range(N)]

combined_transitions = transitions_pos + transitions_F
combined_zero_loss_counts = edge_zero_loss_count_pos + edge_zero_loss_count_F

if plot == "only_combined":
    fig, axes = plt.subplots(
        1, 1, figsize=(7.8, 4.8), sharey=True, constrained_layout=True,
    )
    plot_data = ((combined_transitions, combined_zero_loss_counts, "Combined"),)
else:
    fig, axes = plt.subplots(
            1, 4, figsize=(18, 4.8), sharey=True, constrained_layout=True,
        )
    plot_data = zip(
        (transitions_swp, transitions_pos, transitions_F, combined_transitions),
        (edge_zero_loss_count_swp, edge_zero_loss_count_pos, edge_zero_loss_count_F, combined_zero_loss_counts),
        ("Random", "Position", "Force", "Combined"),
    )

axes = np.atleast_1d(axes)
for ax, (transitions, zero_loss_counts, title) in zip(axes, plot_data):
    # 0: missing, 1: occurred unintentionally, 2: occurred intentionally.
    transition_matrix = np.zeros((N, N), dtype=int)
    for edge in transitions:
        transition_matrix[edge] = 2 if zero_loss_counts[edge] > 0 else 1

    matrix_masked = np.ma.masked_where(np.eye(N, dtype=bool), transition_matrix)
    ax.imshow(
        matrix_masked, cmap=transition_cmap, norm=transition_norm,
        origin="lower", interpolation="none", aspect="equal",
    )
    ax.set_xticks(range(N), labels, rotation=90)
    ax.set_yticks(range(N), labels)
    ax.set_xlabel("final", fontsize=font_size)
    ax.set_title(title, fontsize=font_size, pad=9)
    ax.tick_params(axis="both", labelsize=font_size-2)


axes[0].set_ylabel("initial", fontsize=font_size)
for ax in axes[1:]:
    ax.tick_params(axis="y", left=False, labelleft=False)

legend_handles = [
    patches.Patch(facecolor=color_intentional, label="Intentional"),
    patches.Patch(facecolor=color_unintentional, label="intentional"),
    patches.Patch(facecolor=red, label="Did not occur"),
]
axes[-1].legend(
    handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
    fontsize=font_size, frameon=False, borderaxespad=0,
)

if save is not None:
    if save.lower() != "pdf":
        raise ValueError('plot_arm currently supports only save="pdf".')
    filename = (f"success_{plot}.pdf")
    fig.savefig(filename, format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
importlib.reload(plot_funcs)
export_png = True
export_eps = False
export_pdf = True

fig, axes = plot_funcs.plot_ring_transition_diagrams(
    transitions_pos,
    transitions_F,
    transitions_swp,
    edge_zero_loss_counts=(edge_zero_loss_count_pos, edge_zero_loss_count_F, edge_zero_loss_count_swp),
    missing_edges=(missing_edges_pos, missing_edges_F, missing_edges_swp),
    titles=("Random", "Position", "Force", "Combined"),
    transitions_between_runs=only_init_and_final_buckles,
    only_reached_nodes=only_reached_nodes,
    save_stem=Path("Figures") / "ring_transition_diagrams",
    export_png=export_png,
    export_eps=export_eps,
    export_pdf=export_pdf,
    dpi=600,
    font_size=font_size,
)

## Accuracy a.f.o. H + loss and Hamming improvement

In [ ]:
import numpy as np

H = np.array([4, 5, 6])
accuracy = np.array([0.9, 0.64, 0.34])

folder_H5 = Path("Training\\July10_and_12_H5_safetyMargin_pos_and_recip")
# folder_H5_reciprocal = Path("Training\\July12_reciprocalTo_July10_H5_safetyMargin")
folder_H6 = Path("Training\\July19_H6_pos_pt25LEMargin")

# loss_H5_a, Hamming_H5_a, buckle_pairs_H5_a = file_funcs.build_loss_columns(folder_H5, include_symm=True)
# loss_H5_b, Hamming_H5_b, buckle_pairs_H5_b = file_funcs.build_loss_columns(folder_H5_reciprocal)
# loss_H5 = np.vstack((loss_H5_a, loss_H5_b))
# Hamming_H5 = np.vstack((Hamming_H5_a, Hamming_H5_b))
# buckle_pairs_H5 = np.vstack((buckle_pairs_H5_a, buckle_pairs_H5_b))
loss_H5_nosymm, Hamming_H5_nosymm, _ = file_funcs.build_loss_columns(folder_H5, include_symm=False)
loss_H6_nosymm, Hamming_H6_nosymm, _ = file_funcs.build_loss_columns(folder_H6, include_symm=False)
loss_H5, Hamming_H5, buckle_pairs_H5 = file_funcs.build_loss_columns(folder_H5, include_symm=True)
loss_H6, Hamming_H6, buckle_pairs_H6 = file_funcs.build_loss_columns(folder_H6, include_symm=True)


print('loss initial t final H=5', [np.mean(loss_H5_nosymm, axis=0)[0], np.mean(loss_H5, axis=0)[1]])
print('loss initial t final H=6', [np.mean(loss_H6_nosymm, axis=0)[0], np.mean(loss_H6, axis=0)[1]])

print('Hamming initial t final H=5', [np.mean(Hamming_H5_nosymm, axis=0)[0], np.mean(Hamming_H5, axis=0)[1]])
print('Hamming initial t final H=6', [np.mean(Hamming_H6_nosymm, axis=0)[0], np.mean(Hamming_H6, axis=0)[1]])



In [ ]:
importlib.reload(plot_funcs)

export_png = True
export_pdf = True
save_stem = Path("Figures") / "accuracy_loss_hamming_summary"

fig, axes = plot_funcs.plot_accuracy_loss_hamming_summary(
    H,
    accuracy,
    loss_columns=(loss_H5, loss_H6),
    hamming_columns=(Hamming_H5, Hamming_H6),
    metric_Hs=(5, 6),
    save_path=save_stem.with_suffix(".png") if export_png else None,
    dpi=600,
)

if export_pdf:
    with plt.rc_context({"pdf.fonttype": 42}):
        fig.savefig(save_stem.with_suffix(".pdf"), format="pdf", bbox_inches="tight")

## Chain single position

In [ ]:
# Setup
importlib.reload(plot_funcs)

custom_x_lim_inv = [-0.32, 0.02]
custom_x_lim = [-0.02, 0.26]
custom_y_lim = [-0.14, 0.14]

# 0001 t=1 flat
pos = np.loadtxt("../paper/Setup/Setup data/t=1_0001.csv", delimiter=',')
buckle = [[-1.0],[-1.0],[-1.0],[1.0]]
plot_funcs.plot_arm(np.asarray(pos), -np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = True, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

# 0001 t=2 update
pos = np.loadtxt("../paper/Setup/Setup data/t=2_0001.csv", delimiter=',')
plot_funcs.plot_arm(np.asarray(pos), -np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = True, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

# 0000 t=3 update 
pos = np.loadtxt("../paper/Setup/Setup data/t=3_0000.csv", delimiter=',')
buckle = [[-1.0],[-1.0],[-1.0],[-1.0]]
plot_funcs.plot_arm(np.asarray(pos), -np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = True, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

# 0000 t=4 flat 
pos = np.loadtxt("../paper/Setup/Setup data/t=4_0000.csv", delimiter=',')
plot_funcs.plot_arm(np.asarray(pos), -np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = True, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

In [ ]:
# Train through position
custom_x_lim_inv = [-0.32, 0.02]
custom_x_lim = [-0.02, 0.32]
custom_y_lim = [-0.17, 0.17]

# 0001 free
pos = [[0.0,0.0],[47.20000076293945,0.0],[86.11345672607422,26.7137451171875],[103.02783966064453,70.77896881103516],[91.92655181884766,116.65444946289062],[108.93421936035156,160.6839599609375]]
buckle = [[1.0],[1.0],[1.0],[-1.0]]
plot_funcs.plot_arm(np.asarray(pos)/1000, np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = False, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

# 0110 free
pos=[[0.0,0.0],[47.20000076293945,0.0],[86.01383972167969,-26.85758399963379],[133.2138214111328,-26.847137451171875],[172.0168914794922,0.02590300887823105],[219.21688842773438,0.03968411311507225]]
buckle= [[-1.0],[1.0],[1.0],[-1.0]]
plot_funcs.plot_arm(np.asarray(pos)/1000, np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = False, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

# 1000 free
pos = [[0.0,0.0],[47.20000076293945,0.0],[86.00587463378906,26.869125366210938],[133.2058563232422,26.88309669494629],[172.02618408203125,0.03494499623775482],[188.66749572753906,-44.13412094116211]]
buckle= [[1.0],[-1.0],[-1.0],[-1.0]]
plot_funcs.plot_arm(np.asarray(pos)/1000, np.asarray(buckle), 0.0472, modality=None, show=False, invert_x=False, invert_y = False, 
                    annotate_tip=False, x_lim=custom_x_lim, y_lim=custom_y_lim, dpi = 300, font_size=20, save="pdf")

In [ ]:
importlib.reload(plot_funcs)

# # flat
# pos = [[0.0,0.0],[47.20000076293945,0.0],[94.15695190429688,-6.716611862182617],[141.51495361328125,-3.87457013130188],[188.8000030517578,0.0],[236.0,0.0]]
# buckle = [[-1], [1], [1], [1]]

# # y=40, theta=-1
# pos = [[0.0,0.0],[47.20000076293945,0.0],[89.28108215332031,20.784072875976562],[46.260311126708984,40.66094970703125],[45.281761169433594,87.76535034179688],[70.80000305175781,48.05818176269531]]
# buckle = [[-1],[1],[1],[-1]]

# # y=141 theta = 0.01
# pos = [[0.0,0.0],[47.20000076293945,0.0],[17.2000732421875,-36.4984130859375],[64.24827575683594,-35.698062896728516],[94.88042449951172,0.14819391071796417],[141.59999084472656,6.865454196929932]]
# buckle = [[-1],[1],[1],[1]]

# y=7- theta = -90
# pos = [[0.0,0.0],[47.20000076293945,0.0],[26.786529541015625,-42.58549880981445],[23.98324966430664,-89.7068862915039],[70.80000305175781,-95.25818634033203],[70.80000305175781,-48.05818176269531]]
# buckle = [[-1],[1],[1],[1]]

# # y=6, theta = -8
# pos = [[0.0,0.0],[47.20000076293945,0.0],[49.56996536254883,-47.14029312133789],[94.69303131103516,-33.61190414428711],[94.88042449951172,13.582715034484863],[141.59999084472656,6.865454196929932]]
# buckle = [[-1],[1],[1],[-1]]

# # y=4 theta = -73
# pos = [[0.0,0.0],[47.20000076293945,0.0],[85.55986785888672,27.216777801513672],[43.25975036621094,48.448089599609375],[57.50223159790039,93.34625244140625],[70.80000305175781,48.05818176269531]]
# buckle = [[-1],[1],[-1],[-1]]


## Stress Strain

In [ ]:
import pandas as pd 
file_path = "../Meca500/data/Stress Strain/12Mar/Stress_Strain_Mar12_dl90_3.csv"
exp_df = pd.read_csv(file_path)
plt.plot(exp_df['theta'], -exp_df['F_x'])
plt.xlabel('tip pos')
plt.ylabel('Fx')
plt.show()
# file_funcs.import_stress_strain_exp_and_plot(file_name, plot=True)